<a href="https://colab.research.google.com/github/Ayseatci/DI725_Assignment1/blob/dev/notebooks/Base_RoBERTa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Base RoBERTa Model

This notebook consists of the initial experiments of the base RoBERTa model that only utilizes text/conversation in the dataset for sentiment prediction.

In [4]:
!git clone -b dev https://github.com/Ayseatci/DI725_Assignment1.git

Cloning into 'DI725_Assignment1'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 111 (delta 56), reused 61 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 1002.44 KiB | 7.06 MiB/s, done.
Resolving deltas: 100% (56/56), done.


In [5]:
%cd DI725_Assignment1

/content/DI725_Assignment1


Head truncation mode base RoBERTa experiment and results

In [6]:
!pip install wandb
!wandb login
import wandb

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayseatci00 (ayseatci00-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
import re
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import random
import os
import wandb

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    # CUBLAS determinism
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to: {seed}")

set_seed(SEED)

# Configuration
TRUNCATION_MODE = 'head'
print(f"--- Running experiment with TRUNCATION_MODE: {TRUNCATION_MODE} ---")

# Initialize W&B
wandb.init(
    project="DI725_Assignment_1",
    name=f"base-roberta-{TRUNCATION_MODE}-seed{SEED}",
    config={
        "model": "roberta-base",
        "truncation": TRUNCATION_MODE,
        "epochs": 5,
        "lr": 2e-5,
        "seed": SEED
    }
)

# Load presplit data
train_df = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
val_df = pd.read_csv("data/preprocessed/preprocessed_val.csv").dropna(subset=['text', 'label'])

id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {v: k for k, v in id2label.items()}

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Tokenizer and special tokens
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
new_tokens = ["<cust>", "<agent>"]
tokenizer.add_tokens(new_tokens)

# Tokenization function
def tokenize_experiment(examples):
    outputs = tokenizer(examples["text"], add_special_tokens=False)
    input_ids = []
    attention_masks = []

    for ids in outputs["input_ids"]:
        if len(ids) > 510:
            if TRUNCATION_MODE == 'head':
                processed = ids[:510]
            elif TRUNCATION_MODE == 'tail':
                processed = ids[-510:]
            else:
                processed = ids[:255] + ids[-255:]
            combined = [tokenizer.bos_token_id] + processed + [tokenizer.eos_token_id]
        else:
            combined = [tokenizer.bos_token_id] + ids + [tokenizer.eos_token_id]

        input_ids.append(combined)
        attention_masks.append([1] * len(combined))

    return {"input_ids": input_ids, "attention_mask": attention_masks}

train_dataset = train_dataset.map(tokenize_experiment, batched=True)
val_dataset = val_dataset.map(tokenize_experiment, batched=True)

# Model Initialization
model = RobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)
model.resize_token_embeddings(len(tokenizer))

# Training setup
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

# Training arguments
training_args = TrainingArguments(
    output_dir=f"./roberta_{TRUNCATION_MODE}",
    seed=SEED,
    data_seed=SEED,
    full_determinism=True,
    dataloader_num_workers=0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="wandb"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

# Experiment execution
trainer.train()

# Final Evaluation
print(f"\nFinal Classification Report ({TRUNCATION_MODE}):")
preds = trainer.predict(val_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(preds.label_ids, y_pred, target_names=list(id2label.values())))

wandb.finish()

Global seed set to: 42
--- Running experiment with TRUNCATION_MODE: head ---


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ayseatci00 (ayseatci00-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Classification Report (head):


              precision    recall  f1-score   support

    negative       0.94      0.91      0.93        82
     neutral       0.91      0.95      0.93       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.92       194
   macro avg       0.62      0.62      0.62       194
weighted avg       0.91      0.92      0.92       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▁▄▅██
eval/f1_macro,▁▅▅██
eval/loss,█▂▅▃▁
eval/runtime,▃▃▁▄█
eval/samples_per_second,▆▆█▅▁
eval/steps_per_second,▆▆█▅▁
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


Tail truncation mode base RoBERTa experiment and results

In [8]:
TRUNCATION_MODE = 'tail'

set_seed(SEED)
torch.cuda.empty_cache()

# Reinitialize WB
wandb.init(
    project="DI725_Assignment_1",
    name=f"base-roberta-{TRUNCATION_MODE}-seed{SEED}",
    config={
        "model": "roberta-base",
        "truncation": TRUNCATION_MODE,
        "epochs": 5,
        "lr": 2e-5,
        "seed": SEED
    },
    reinit=True
)

# Remap the datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize_experiment, batched=True)
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True)).map(tokenize_experiment, batched=True)


model = RobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)
model.resize_token_embeddings(len(tokenizer))

training_args.output_dir = f"./roberta_{TRUNCATION_MODE}" # output path
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

# Final Evaluation
print(f"\nFinal Classification Report ({TRUNCATION_MODE}):")
preds = trainer.predict(val_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(preds.label_ids, y_pred, target_names=list(id2label.values())))
wandb.finish()

Global seed set to: 42


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Classification Report (tail):


              precision    recall  f1-score   support

    negative       0.94      0.90      0.92        82
     neutral       0.90      0.95      0.93       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.92       194
   macro avg       0.61      0.62      0.62       194
weighted avg       0.90      0.92      0.91       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▁█▇██
eval/f1_macro,▁█▇██
eval/loss,█▁▂▃▃
eval/runtime,▇█▃▁▄
eval/samples_per_second,▂▁▆█▅
eval/steps_per_second,▂▁▆█▅
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


Mixed truncation mode base RoBERTa experiment and results

In [9]:
TRUNCATION_MODE = 'mixed'

set_seed(SEED)
torch.cuda.empty_cache()

# Reinitialize WB
wandb.init(
    project="DI725_Assignment_1",
    name=f"base-roberta-{TRUNCATION_MODE}-seed{SEED}",
    config={
        "model": "roberta-base",
        "truncation": TRUNCATION_MODE,
        "epochs": 5,
        "lr": 2e-5,
        "seed": SEED
    },
    reinit=True
)

# Remap the datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize_experiment, batched=True)
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True)).map(tokenize_experiment, batched=True)


model = RobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)
model.resize_token_embeddings(len(tokenizer))

training_args.output_dir = f"./roberta_{TRUNCATION_MODE}" # output path
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

# Final Evaluation
print(f"\nFinal Classification Report ({TRUNCATION_MODE}):")
preds = trainer.predict(val_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(preds.label_ids, y_pred, target_names=list(id2label.values())))
wandb.finish()

Global seed set to: 42


Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Classification Report (mixed):


              precision    recall  f1-score   support

    negative       0.93      0.93      0.93        82
     neutral       0.92      0.94      0.93       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.92       194
   macro avg       0.62      0.62      0.62       194
weighted avg       0.91      0.92      0.92       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▁▇▆██
eval/f1_macro,▁▇▆██
eval/loss,█▁▄▁▂
eval/runtime,▁█▃▄▁
eval/samples_per_second,█▁▅▅█
eval/steps_per_second,█▁▅▅█
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


The experiment results of the based RoBERTa model with different truncation modes show that the textual context alone is insufficient for sentiment classification in this dataset.

Neutral and positive texts appear to have high lexical overlap therefore a pure RoBERTa model cannot distinguish between these two classes which can be seen by the consistent 0.00 F1-score and recall for the positive class across different truncation strategies.

While head truncation provided slightly higher precision for negative cases, and tail truncation improved overall macro F1, the model consistently fails to capture the signals of positive sentiment.

As a result, a pure transformer based approach is inadequate for the imbalanced dataset. To solve this problem, additional features such as the numerical and categorical features should be integrated into a hybrid architecture so that the model can support the context.

"Although Head truncation yielded a marginally higher peak F1-Macro ($0.619$ vs $0.615$), Tail truncation was selected for further experiments due to superior loss convergence. Tail truncation achieved a lower Validation Loss ($0.318$) and reached its performance plateau significantly faster (Epoch 2 vs Epoch 5). This suggests that the concluding dialogue turns provide a cleaner, less noisy signal for sentiment classification, whereas the introductory turns (Head) contain boilerplate language that leads to slower convergence and higher validation error.

## Additional experiments with tail truncation configuration

Experiment with focal loss

In [10]:
from torch import nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight

SEED = 42

set_seed(SEED)

# Experiment Settings
EXPERIMENT_TYPE = 'focal' # 'focal','oversample', 'downsample', or 'weighted'
PROJECT_NAME = "DI725_Assignment_1"
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-seed{SEED}"

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

# Explicit Data Loading
train_df = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
val_df = pd.read_csv("data/preprocessed/preprocessed_val.csv").dropna(subset=['text', 'label'])

id2label = {0: "negative", 1: "neutral", 2: "positive"}

# Apply sampling to train only
if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    max_size = counts.max()
    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)
        if n_current == max_size:
            lst.append(train_df[train_df['label'] == i])
        else:
            target_size = min(n_current * 4, max_size)
            if n_current > 0:
                upsampled = train_df[train_df['label'] == i].sample(
                    target_size, replace=True, random_state=SEED
                )
                lst.append(upsampled)
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

elif EXPERIMENT_TYPE == 'downsample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()
    multiplier = 10
    target_cap = min_size * multiplier
    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)
        if n_current <= target_cap:
            lst.append(train_df[train_df['label'] == i])
        else:
            lst.append(train_df[train_df['label'] == i].sample(target_cap, random_state=SEED))
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

# Convert to Dataset objects
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Focal loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class ImbalanceTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = compute_class_weight(
            'balanced',
            classes=np.unique(train_df['label']),
            y=train_df['label']
        )
        class_weights = torch.tensor(weights, dtype=torch.float).to(model.device)

        if EXPERIMENT_TYPE == 'weighted':
            loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        elif EXPERIMENT_TYPE == 'focal':
            loss_fct = FocalLoss(alpha=class_weights, gamma=2.0)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Training execution
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
tokenizer.add_tokens(["<cust>", "<agent>"])

def tokenize_tail(examples):
    outputs = tokenizer(examples["text"], add_special_tokens=False)
    input_ids = []
    for ids in outputs["input_ids"]:
        processed = ids[-510:] if len(ids) > 510 else ids
        input_ids.append([tokenizer.bos_token_id] + processed + [tokenizer.eos_token_id])
    return {"input_ids": input_ids}

train_dataset = train_dataset.map(tokenize_tail, batched=True)
val_dataset = val_dataset.map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

# Training arguments
training_args = TrainingArguments(
    output_dir=f"./results_{EXPERIMENT_TYPE}",
    eval_strategy="epoch",
    logging_steps=10,
    report_to="wandb",
    learning_rate=2e-5,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    seed=SEED,
    data_seed=SEED,
    full_determinism=True,
    dataloader_num_workers=0,
    fp16=torch.cuda.is_available()
)

trainer = ImbalanceTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()


print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")

preds_output = trainer.predict(val_dataset)
logits = preds_output.predictions
y_true = preds_output.label_ids
y_pred = np.argmax(logits, axis=1)

report = classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))]
)
print(report)

wandb.log({"final_classification_report": report})

wandb.finish()

Global seed set to: 42


Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (520 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.716838,0.920251,0.783505,0.510041
2,0.966924,0.578749,0.871134,0.583542
3,0.461935,0.596491,0.896907,0.601122
4,0.142821,0.488117,0.896907,0.603957
5,0.153626,0.478863,0.886598,0.600588


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: focal (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.89      0.91      0.90        82
     neutral       0.91      0.89      0.90       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.89       194
   macro avg       0.60      0.60      0.60       194
weighted avg       0.89      0.89      0.89       194



eval/accuracy,▁▆██▇
eval/f1_macro,▁▆███
eval/loss,█▃▃▁▁
eval/runtime,▇▁▆██
eval/samples_per_second,▂█▃▁▁
eval/steps_per_second,▂█▃▁▁
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


Weighted Loss

In [11]:
# Weighted cross entropy
# Update Experiment Settings
EXPERIMENT_TYPE = 'weighted'
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-seed{SEED}"

set_seed(SEED)
torch.cuda.empty_cache()

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

train_df = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
val_df = pd.read_csv("data/preprocessed/preprocessed_val.csv").dropna(subset=['text', 'label'])

# Tokenize Datasets
train_dataset = Dataset.from_pandas(train_df).map(tokenize_tail, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_tail, batched=True)

# Reinitialize model
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

# Update Trainer Arguments and Run
training_args.output_dir = f"./results_{EXPERIMENT_TYPE}"

trainer = ImbalanceTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

# Evaluation and Report
print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
report = classification_report(preds_output.label_ids, y_pred, target_names=list(id2label.values()))
print(report)

wandb.log({"final_classification_report": report})
wandb.finish()

Global seed set to: 42


Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: weighted (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.91      0.88      0.89        82
     neutral       0.89      0.94      0.91       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.90       194
   macro avg       0.60      0.60      0.60       194
weighted avg       0.88      0.90      0.89       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▁▅▇█▇
eval/f1_macro,▁▅▇█▇
eval/loss,▄▁▅▆█
eval/runtime,▃▁█▅▃
eval/samples_per_second,▆█▁▃▆
eval/steps_per_second,▆█▁▃▆
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


Experiment with oversample

In [13]:
# Update Experiment Mode
EXPERIMENT_TYPE = 'oversample'
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-mult3-seed{SEED}"

set_seed(SEED)
torch.cuda.empty_cache()

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

train_df = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
val_df = pd.read_csv("data/preprocessed/preprocessed_val.csv").dropna(subset=['text', 'label'])

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == min_size:
            max_size = counts.max()
            target_size = min(n_current * 3, max_size)

            upsampled = train_df[train_df['label'] == i].sample(
                target_size,
                replace=True,
                random_state=SEED
            )
            lst.append(upsampled)
            print(f"Upsampling Class {i} from {n_current} to {target_size}")
        else:
            lst.append(train_df[train_df['label'] == i])

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

elif EXPERIMENT_TYPE == 'downsample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()
    multiplier = 10 # 10x Multiplier
    target_cap = min_size * multiplier
    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)
        if n_current <= target_cap:
            lst.append(train_df[train_df['label'] == i])
        else:
            downsampled = train_df[train_df['label'] == i].sample(
                target_cap, random_state=SEED
            )
            lst.append(downsampled)
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Final training size for {EXPERIMENT_TYPE}: {len(train_df)}")
print(train_df['label'].value_counts())

train_dataset = Dataset.from_pandas(train_df).map(tokenize_tail, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args.output_dir = f"./results_{EXPERIMENT_TYPE}-mult3"

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
report = classification_report(preds_output.label_ids, y_pred, target_names=list(id2label.values()))
print(report)

wandb.log({"final_classification_report": report})
wandb.finish()

Global seed set to: 42


Upsampling Class 2 from 14 to 42
Final training size for oversample: 801
label
1    433
0    326
2     42
Name: count, dtype: int64


Map:   0%|          | 0/801 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: oversample (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.89      0.90      0.90        82
     neutral       0.90      0.92      0.91       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.90       194
   macro avg       0.60      0.61      0.60       194
weighted avg       0.88      0.90      0.89       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▁▆█▇█
eval/f1_macro,▁▅█▆▇
eval/loss,▃▁▂█▇
eval/runtime,▁▃▁▂█
eval/samples_per_second,█▅█▇▁
eval/steps_per_second,█▅█▇▁
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


In [14]:
# Update Experiment Mode
EXPERIMENT_TYPE = 'oversample'
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-mult4-seed{SEED}"

set_seed(SEED)
torch.cuda.empty_cache()

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

train_df = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
val_df = pd.read_csv("data/preprocessed/preprocessed_val.csv").dropna(subset=['text', 'label'])

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == min_size:
            max_size = counts.max()
            target_size = min(n_current * 4, max_size)

            upsampled = train_df[train_df['label'] == i].sample(
                target_size,
                replace=True,
                random_state=SEED
            )
            lst.append(upsampled)
            print(f"Upsampling Class {i} from {n_current} to {target_size}")
        else:
            lst.append(train_df[train_df['label'] == i])

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

elif EXPERIMENT_TYPE == 'downsample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()
    multiplier = 10 # 10x Multiplier
    target_cap = min_size * multiplier
    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)
        if n_current <= target_cap:
            lst.append(train_df[train_df['label'] == i])
        else:
            downsampled = train_df[train_df['label'] == i].sample(
                target_cap, random_state=SEED
            )
            lst.append(downsampled)
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Final training size for {EXPERIMENT_TYPE}: {len(train_df)}")
print(train_df['label'].value_counts())

train_dataset = Dataset.from_pandas(train_df).map(tokenize_tail, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args.output_dir = f"./results_{EXPERIMENT_TYPE}-mult4"

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
report = classification_report(preds_output.label_ids, y_pred, target_names=list(id2label.values()))
print(report)

wandb.log({"final_classification_report": report})
wandb.finish()

Global seed set to: 42


Upsampling Class 2 from 14 to 56
Final training size for oversample: 815
label
1    433
0    326
2     56
Name: count, dtype: int64


Map:   0%|          | 0/815 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: oversample (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.91      0.88      0.89        82
     neutral       0.89      0.93      0.91       109
    positive       0.50      0.33      0.40         3

    accuracy                           0.90       194
   macro avg       0.77      0.71      0.73       194
weighted avg       0.90      0.90      0.90       194



eval/accuracy,▁▅▇██
eval/f1_macro,▁▂▃▃█
eval/loss,█▁▆▄▅
eval/runtime,▃▁█▂▅
eval/samples_per_second,▆█▁▇▄
eval/steps_per_second,▆█▁▇▄
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


In [15]:
# Update Experiment Mode
EXPERIMENT_TYPE = 'oversample'
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-mult6-seed{SEED}"

set_seed(SEED)
torch.cuda.empty_cache()

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

train_df = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
val_df = pd.read_csv("data/preprocessed/preprocessed_val.csv").dropna(subset=['text', 'label'])

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == min_size:
            max_size = counts.max()
            target_size = min(n_current * 6, max_size)

            upsampled = train_df[train_df['label'] == i].sample(
                target_size,
                replace=True,
                random_state=SEED
            )
            lst.append(upsampled)
            print(f"Upsampling Class {i} from {n_current} to {target_size}")
        else:
            lst.append(train_df[train_df['label'] == i])

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

elif EXPERIMENT_TYPE == 'downsample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()
    multiplier = 10 # 10x Multiplier
    target_cap = min_size * multiplier
    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)
        if n_current <= target_cap:
            lst.append(train_df[train_df['label'] == i])
        else:
            downsampled = train_df[train_df['label'] == i].sample(
                target_cap, random_state=SEED
            )
            lst.append(downsampled)
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Final training size for {EXPERIMENT_TYPE}: {len(train_df)}")
print(train_df['label'].value_counts())

train_dataset = Dataset.from_pandas(train_df).map(tokenize_tail, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args.output_dir = f"./results_{EXPERIMENT_TYPE}-mult6"

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
report = classification_report(preds_output.label_ids, y_pred, target_names=list(id2label.values()))
print(report)

wandb.log({"final_classification_report": report})
wandb.finish()

Global seed set to: 42


Upsampling Class 2 from 14 to 84
Final training size for oversample: 843
label
1    433
0    326
2     84
Name: count, dtype: int64


Map:   0%|          | 0/843 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: oversample (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.91      0.87      0.89        82
     neutral       0.89      0.93      0.91       109
    positive       0.50      0.33      0.40         3

    accuracy                           0.89       194
   macro avg       0.77      0.71      0.73       194
weighted avg       0.89      0.89      0.89       194



eval/accuracy,▁█▅██
eval/f1_macro,▁▂▇▇█
eval/loss,▃▁▅██
eval/runtime,▁▁▁▁█
eval/samples_per_second,████▁
eval/steps_per_second,████▁
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


In [16]:
# Update Experiment Mode
EXPERIMENT_TYPE = 'oversample'
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-mult8-seed{SEED}"

set_seed(SEED)
torch.cuda.empty_cache()

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

train_df = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
val_df = pd.read_csv("data/preprocessed/preprocessed_val.csv").dropna(subset=['text', 'label'])

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == min_size:
            max_size = counts.max()
            target_size = min(n_current * 8, max_size)

            upsampled = train_df[train_df['label'] == i].sample(
                target_size,
                replace=True,
                random_state=SEED
            )
            lst.append(upsampled)
            print(f"Upsampling Class {i} from {n_current} to {target_size}")
        else:
            lst.append(train_df[train_df['label'] == i])

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

elif EXPERIMENT_TYPE == 'downsample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()
    multiplier = 10 # 10x Multiplier
    target_cap = min_size * multiplier
    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)
        if n_current <= target_cap:
            lst.append(train_df[train_df['label'] == i])
        else:
            downsampled = train_df[train_df['label'] == i].sample(
                target_cap, random_state=SEED
            )
            lst.append(downsampled)
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Final training size for {EXPERIMENT_TYPE}: {len(train_df)}")
print(train_df['label'].value_counts())

train_dataset = Dataset.from_pandas(train_df).map(tokenize_tail, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args.output_dir = f"./results_{EXPERIMENT_TYPE}-mult8"

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
report = classification_report(preds_output.label_ids, y_pred, target_names=list(id2label.values()))
print(report)

wandb.log({"final_classification_report": report})
wandb.finish()

Global seed set to: 42


Upsampling Class 2 from 14 to 112
Final training size for oversample: 871
label
1    433
0    326
2    112
Name: count, dtype: int64


Map:   0%|          | 0/871 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: oversample (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.91      0.89      0.90        82
     neutral       0.90      0.93      0.91       109
    positive       0.50      0.33      0.40         3

    accuracy                           0.90       194
   macro avg       0.77      0.72      0.74       194
weighted avg       0.90      0.90      0.90       194



eval/accuracy,▁▆▇█▇
eval/f1_macro,▁▃█▇▆
eval/loss,█▁▃▃▅
eval/runtime,▁▃▂▁█
eval/samples_per_second,█▅▇█▁
eval/steps_per_second,█▅▇█▁
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


Experiment with downsample

Downsampling multiplier with scale 10

In [17]:
# Update Experiment Mode
EXPERIMENT_TYPE = 'downsample'
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-mult8-seed{SEED}"

set_seed(SEED)
torch.cuda.empty_cache()

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

train_df = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
val_df = pd.read_csv("data/preprocessed/preprocessed_val.csv").dropna(subset=['text', 'label'])

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == min_size:
            max_size = counts.max()
            target_size = min(n_current * 8, max_size)

            upsampled = train_df[train_df['label'] == i].sample(
                target_size,
                replace=True,
                random_state=SEED
            )
            lst.append(upsampled)
            print(f"Upsampling Class {i} from {n_current} to {target_size}")
        else:
            lst.append(train_df[train_df['label'] == i])

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

elif EXPERIMENT_TYPE == 'downsample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()
    multiplier = 8 # 8x Multiplier
    target_cap = min_size * multiplier
    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)
        if n_current <= target_cap:
            lst.append(train_df[train_df['label'] == i])
        else:
            downsampled = train_df[train_df['label'] == i].sample(
                target_cap, random_state=SEED
            )
            lst.append(downsampled)
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Final training size for {EXPERIMENT_TYPE}: {len(train_df)}")
print(train_df['label'].value_counts())

train_dataset = Dataset.from_pandas(train_df).map(tokenize_tail, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args.output_dir = f"./results_{EXPERIMENT_TYPE}-mult8"

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
report = classification_report(preds_output.label_ids, y_pred, target_names=list(id2label.values()))
print(report)

wandb.log({"final_classification_report": report})
wandb.finish()

Global seed set to: 42


Final training size for downsample: 238
label
1    112
0    112
2     14
Name: count, dtype: int64


Map:   0%|          | 0/238 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: downsample (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.83      0.89      0.86        82
     neutral       0.89      0.86      0.87       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.86       194
   macro avg       0.57      0.58      0.58       194
weighted avg       0.85      0.86      0.85       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▁▅▇██
eval/f1_macro,▁▅▇██
eval/loss,█▆▃▂▁
eval/runtime,█▁▁▃█
eval/samples_per_second,▁██▆▁
eval/steps_per_second,▁██▆▁
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


Downsampling multiplier with scale 6

In [18]:
# Update Experiment Mode
EXPERIMENT_TYPE = 'downsample'
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-mult6-seed{SEED}"

set_seed(SEED)
torch.cuda.empty_cache()

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

train_df = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
val_df = pd.read_csv("data/preprocessed/preprocessed_val.csv").dropna(subset=['text', 'label'])

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == min_size:
            max_size = counts.max()
            target_size = min(n_current * 8, max_size)

            upsampled = train_df[train_df['label'] == i].sample(
                target_size,
                replace=True,
                random_state=SEED
            )
            lst.append(upsampled)
            print(f"Upsampling Class {i} from {n_current} to {target_size}")
        else:
            lst.append(train_df[train_df['label'] == i])

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

elif EXPERIMENT_TYPE == 'downsample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()
    multiplier = 6 # 6x Multiplier
    target_cap = min_size * multiplier
    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)
        if n_current <= target_cap:
            lst.append(train_df[train_df['label'] == i])
        else:
            downsampled = train_df[train_df['label'] == i].sample(
                target_cap, random_state=SEED
            )
            lst.append(downsampled)
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Final training size for {EXPERIMENT_TYPE}: {len(train_df)}")
print(train_df['label'].value_counts())

train_dataset = Dataset.from_pandas(train_df).map(tokenize_tail, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args.output_dir = f"./results_{EXPERIMENT_TYPE}-mult6"

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
report = classification_report(preds_output.label_ids, y_pred, target_names=list(id2label.values()))
print(report)

wandb.log({"final_classification_report": report})
wandb.finish()

Global seed set to: 42


Final training size for downsample: 182
label
0    84
1    84
2    14
Name: count, dtype: int64


Map:   0%|          | 0/182 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: downsample (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.85      0.84      0.85        82
     neutral       0.86      0.89      0.87       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.86       194
   macro avg       0.57      0.58      0.57       194
weighted avg       0.84      0.86      0.85       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▂▁▄▆█
eval/f1_macro,▄▁▅▇█
eval/loss,██▄▃▁
eval/runtime,▁▁██▃
eval/samples_per_second,██▁▁▆
eval/steps_per_second,██▁▁▆
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


Downsampling multiplier with scale 4

In [19]:
# Update Experiment Mode
EXPERIMENT_TYPE = 'downsample'
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-mult4-seed{SEED}"

set_seed(SEED)
torch.cuda.empty_cache()

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

train_df = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
val_df = pd.read_csv("data/preprocessed/preprocessed_val.csv").dropna(subset=['text', 'label'])

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == min_size:
            max_size = counts.max()
            target_size = min(n_current * 8, max_size)

            upsampled = train_df[train_df['label'] == i].sample(
                target_size,
                replace=True,
                random_state=SEED
            )
            lst.append(upsampled)
            print(f"Upsampling Class {i} from {n_current} to {target_size}")
        else:
            lst.append(train_df[train_df['label'] == i])

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

elif EXPERIMENT_TYPE == 'downsample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()
    multiplier = 4 # 4x Multiplier
    target_cap = min_size * multiplier
    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)
        if n_current <= target_cap:
            lst.append(train_df[train_df['label'] == i])
        else:
            downsampled = train_df[train_df['label'] == i].sample(
                target_cap, random_state=SEED
            )
            lst.append(downsampled)
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Final training size for {EXPERIMENT_TYPE}: {len(train_df)}")
print(train_df['label'].value_counts())

train_dataset = Dataset.from_pandas(train_df).map(tokenize_tail, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args.output_dir = f"./results_{EXPERIMENT_TYPE}-mult4"

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
report = classification_report(preds_output.label_ids, y_pred, target_names=list(id2label.values()))
print(report)

wandb.log({"final_classification_report": report})
wandb.finish()

Global seed set to: 42


Final training size for downsample: 126
label
1    56
0    56
2    14
Name: count, dtype: int64


Map:   0%|          | 0/126 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: downsample (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.60      0.63      0.62        82
     neutral       0.69      0.68      0.69       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.65       194
   macro avg       0.43      0.44      0.43       194
weighted avg       0.64      0.65      0.65       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▁█▂▃▂
eval/f1_macro,▂█▁▄▂
eval/loss,█▇▄▂▁
eval/runtime,▅▄▁▆█
eval/samples_per_second,▄▅█▃▁
eval/steps_per_second,▄▅█▃▁
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


Downsampling multiplier with scale 3

In [20]:
# Update Experiment Mode
EXPERIMENT_TYPE = 'downsample'
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-mult3-seed{SEED}"

set_seed(SEED)
torch.cuda.empty_cache()

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

train_df = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
val_df = pd.read_csv("data/preprocessed/preprocessed_val.csv").dropna(subset=['text', 'label'])

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == min_size:
            max_size = counts.max()
            target_size = min(n_current * 8, max_size)

            upsampled = train_df[train_df['label'] == i].sample(
                target_size,
                replace=True,
                random_state=SEED
            )
            lst.append(upsampled)
            print(f"Upsampling Class {i} from {n_current} to {target_size}")
        else:
            lst.append(train_df[train_df['label'] == i])

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

elif EXPERIMENT_TYPE == 'downsample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()
    multiplier = 3 # 3x Multiplier
    target_cap = min_size * multiplier
    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)
        if n_current <= target_cap:
            lst.append(train_df[train_df['label'] == i])
        else:
            downsampled = train_df[train_df['label'] == i].sample(
                target_cap, random_state=SEED
            )
            lst.append(downsampled)
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Final training size for {EXPERIMENT_TYPE}: {len(train_df)}")
print(train_df['label'].value_counts())

train_dataset = Dataset.from_pandas(train_df).map(tokenize_tail, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args.output_dir = f"./results_{EXPERIMENT_TYPE}-mult3"

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
report = classification_report(preds_output.label_ids, y_pred, target_names=list(id2label.values()))
print(report)

wandb.log({"final_classification_report": report})
wandb.finish()

Global seed set to: 42


Final training size for downsample: 98
label
1    42
0    42
2    14
Name: count, dtype: int64


Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: downsample (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.60      0.70      0.64        82
     neutral       0.72      0.65      0.68       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.66       194
   macro avg       0.44      0.45      0.44       194
weighted avg       0.66      0.66      0.66       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▁████
eval/f1_macro,▁████
eval/loss,█▅▁▄▃
eval/runtime,▂▁▂▁█
eval/samples_per_second,▇█▇█▁
eval/steps_per_second,▇█▇█▁
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


## Summary of Base RoBERTa Model Experiments

Tail truncation had a slightly larger macro F1 therefore was chosen as the base model for further experience. This result is also consistent with the .. that the resolution or main sentiment conclusion happens at the end of conversations therefore the beginning of dialgues are usully noises.

In addiiton to these experiments, the weighted loss or focal loss has not contributed to the model. Oversampling has been observed to increase the recall of the positive classes. Therefore adiditional feature engineering practices will be performed on the tail truncated model with oversampling strategy.